# Tourism Nights Distribution Across Europe
### Data analysis notebook · DataBites · Josep Ferrer

---

This notebook documents the full data pipeline behind the story
**"3 billion tourist nights in 2024. Half went to the same 20 regions."**

**The story in one sentence:** Tourist nights across Europe are heavily
concentrated in a handful of coastal and capital regions, and while the
2020 COVID crash was brutal and uniform, the recovery has been unequal.

**Data source:** EU Tourism Dashboard, dataset `TOUR_NIGHT_SPENT`
via the territorial.ec.europa.eu public API. Nights spent at tourist
accommodation establishments by NUTS3 sub-region, 2019 to 2024.

---

### Pipeline overview

```
Tourism Dashboard API  ->  raw CSVs (cached)  ->  clean DataFrames  ->  analysis  ->  CSVs  ->  Datawrapper
```

| Step | What happens |
|------|-------------|
| 0 | Imports, paths, API fetch with local cache |
| 1 | Load and clean the raw data |
| 2 | Explore: COVID crash, concentration, unequal recovery |
| 3 | Export four chart-ready CSVs |

## 0. Setup, fetch and cache

**Caching strategy.** On the first run, both CSVs are fetched from the
Tourism Dashboard API and saved to `data/`. Subsequent runs load from
disk. This keeps the notebook reproducible without repeated API calls.

**Column reference for the raw CSV:**
- `TERRITORY_ID`: NUTS3 code (e.g. `AT111` = Mittelburgenland, Austria)
- `NAME_HTML`: region name
- `YEAR`: reference year (2019 to 2024)
- `VALUE`: absolute number of tourist nights
- `VERSIONS`: boundary edition (2016 or 2021) -- we keep the most recent
- `UNIT`: always `PC` here meaning count (absolute nights, not percentage)

In [1]:
import pandas as pd
import requests
from pathlib import Path
import os

# Run from repo root or notebook folder -- both work
NOTEBOOK_DIR = Path(__file__).parent if "__file__" in dir() else Path.cwd()
# If running from repo root, point into the story folder
if (NOTEBOOK_DIR / "stories").exists():
    STORY_DIR = NOTEBOOK_DIR / "stories" / "tourism_nights_distribution"
else:
    STORY_DIR = NOTEBOOK_DIR

DATA_DIR   = STORY_DIR / "data"
OUTPUT_DIR = STORY_DIR / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

API_BASE = "https://territorial.ec.europa.eu/api/udp/v2/en/data/TOUR_NIGHT_SPENT"

def fetch_and_cache(nutslevel, filename):
    filepath = DATA_DIR / filename
    if filepath.exists():
        print(f"Loading {filename} from cache...")
    else:
        print(f"Fetching {filename} from API (nutslevel={nutslevel})...")
        r = requests.get(API_BASE, params={"nutslevel": str(nutslevel), "format": "csv"}, timeout=60)
        r.raise_for_status()
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(r.text)
        print(f"Saved to {filepath} ({len(r.text)/1024:.0f} KB)")
    return pd.read_csv(filepath)

nuts3 = fetch_and_cache(3, "nights_nuts3.csv")
nuts2 = fetch_and_cache(2, "nights_nuts2_all_years.csv")

print()
print(f"NUTS3 raw shape: {nuts3.shape}")
print(f"NUTS2 raw shape: {nuts2.shape}")
print(f"Columns: {nuts3.columns.tolist()}")

Loading nights_nuts3.csv from cache...
Loading nights_nuts2_all_years.csv from cache...

NUTS3 raw shape: (20994, 8)
NUTS2 raw shape: (4530, 8)
Columns: ['VERSIONS', 'LEVEL_ID', 'TERRITORY_ID', 'NAME_HTML', 'YEAR', 'DATE', 'UNIT', 'VALUE']


## 1. Clean and validate

Three cleaning steps:

**1. Deduplicate boundary versions.** Some regions appear under both the
2016 and 2021 boundary editions. We keep the most recent version per
region-year combination to avoid double-counting.

**2. Extract country code.** The first two characters of `TERRITORY_ID`
are the ISO2 country code (`AT111` -> `AT`). This lets us aggregate by
country later.

**3. Type conversion.** Cast year to integer and nights to float.
`errors="coerce"` silently drops any flagged or missing values.

In [2]:
def clean(df):
    # Keep most recent boundary version per region-year
    df = df.sort_values("VERSIONS", ascending=False)
    df = df.drop_duplicates(subset=["TERRITORY_ID", "YEAR"], keep="first")

    # Extract country code from first 2 characters of NUTS code
    df = df.copy()
    df["country"] = df["TERRITORY_ID"].str[:2]

    # Rename for readability
    df = df.rename(columns={
        "TERRITORY_ID": "nuts_code",
        "NAME_HTML":    "region_name",
        "YEAR":         "year",
        "VALUE":        "nights"
    })

    df = df[["nuts_code", "region_name", "country", "year", "nights"]].copy()
    df["year"]   = df["year"].astype(int)
    df["nights"] = pd.to_numeric(df["nights"], errors="coerce")
    df = df.dropna(subset=["nights"])

    return df.sort_values(["nuts_code", "year"]).reset_index(drop=True)


nuts3_clean = clean(nuts3)
nuts2_clean = clean(nuts2)

print(f"NUTS3 after cleaning: {nuts3_clean.shape}")
print(f"NUTS2 after cleaning: {nuts2_clean.shape}")
print(f"Countries in NUTS3:   {sorted(nuts3_clean['country'].unique())}")
print(f"Years available:      {sorted(nuts3_clean['year'].unique())}")
print()
nuts3_clean.head(6)

NUTS3 after cleaning: (7180, 5)
NUTS2 after cleaning: (1534, 5)
Countries in NUTS3:   ['AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']
Years available:      [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]



,nuts_code,region_name,country,year,nights
0,AT111,Mittelburgenland,AT,2019,359291.0
1,AT111,Mittelburgenland,AT,2020,177094.0
2,AT111,Mittelburgenland,AT,2021,222967.0
3,AT111,Mittelburgenland,AT,2022,340427.0
4,AT111,Mittelburgenland,AT,2023,321841.0
5,AT111,Mittelburgenland,AT,2024,318311.0


## 2. Explore: finding the story

**Question 1:** How did COVID hit and how did the recovery unfold?
The EU aggregate 2019-2024 is the arc of the story.

**Question 2:** Where are the nights concentrated?
The top 20 NUTS3 regions dominate the map.

**Question 3:** Which regions recovered fastest and slowest?
The unequal recovery is the second act of the story.

In [3]:
# Question 1: EU aggregate trend (sum all NUTS3 by year)
eu_trend = nuts3_clean.groupby("year")["nights"].sum().reset_index()
eu_trend["nights_bn"]   = (eu_trend["nights"] / 1e9).round(2)

baseline = eu_trend.loc[eu_trend["year"] == 2019, "nights"].values[0]
eu_trend["vs_2019_pct"] = ((eu_trend["nights"] / baseline) - 1) * 100

print("=== EU total tourist nights 2019-2024 ===")
print(f"{'Year':<6} {'Nights (B)':>12} {'vs 2019':>10}")
print("-" * 32)
for _, row in eu_trend.iterrows():
    print(f"{int(row['year']):<6} {row['nights_bn']:>12.2f} {row['vs_2019_pct']:>+10.1f}%")

=== EU total tourist nights 2019-2024 ===
Year     Nights (B)    vs 2019
--------------------------------
2019           2.91       +0.0%
2020           1.45      -50.2%
2021           1.90      -34.9%
2022           2.86       -1.9%
2023           3.05       +4.7%
2024           3.12       +7.2%


In [4]:
# Question 2: Top 20 NUTS3 regions by nights in 2024
latest_year  = nuts3_clean["year"].max()
total_latest = nuts3_clean[nuts3_clean["year"] == latest_year]["nights"].sum()

top20 = (
    nuts3_clean[nuts3_clean["year"] == latest_year]
    .sort_values("nights", ascending=False)
    .head(20)
    .copy()
)
top20["share_pct"] = (top20["nights"] / total_latest * 100).round(1)

print(f"=== Top 20 NUTS3 regions by nights, {latest_year} ===")
print(f"{'Country':<8} {'Region':<40} {'Nights (M)':>10} {'Share':>7}")
print("-" * 68)
for _, row in top20.iterrows():
    print(f"{row['country']:<8} {row['region_name']:<40} {row['nights']/1e6:>10.1f} {row['share_pct']:>6.1f}%")

=== Top 20 NUTS3 regions by nights, 2024 ===
Country  Region                                   Nights (M)   Share
--------------------------------------------------------------------
ES       Mallorca                                       55.3    1.8%
IT       Roma                                           47.2    1.5%
FR       Paris                                          43.2    1.4%
ES       Barcelona                                      41.2    1.3%
IT       Venezia                                        38.8    1.2%
IT       Bolzano-Bozen                                  37.0    1.2%
ES       Tenerife                                       35.8    1.1%
ES       Alicante / Alacant                             32.6    1.0%
ES       Madrid                                         32.3    1.0%
ES       Málaga                                         30.5    1.0%
DE       Berlin                                         30.4    1.0%
EL       Kalymnos, Karpathos, Kos, Rodos                28

In [5]:
# Concentration: how top-heavy is European tourism?
sorted_2024 = nuts3_clean[nuts3_clean["year"] == latest_year].sort_values("nights", ascending=False)

print("=== Cumulative concentration, 2024 ===")
for n in [10, 20, 50, 100]:
    share = sorted_2024.head(n)["nights"].sum() / total_latest * 100
    print(f"  Top {n:3d} NUTS3 regions: {share:.1f}% of all EU tourist nights")

=== Cumulative concentration, 2024 ===
  Top  10 NUTS3 regions: 12.6% of all EU tourist nights
  Top  20 NUTS3 regions: 20.6% of all EU tourist nights
  Top  50 NUTS3 regions: 35.6% of all EU tourist nights
  Top 100 NUTS3 regions: 50.4% of all EU tourist nights


In [6]:
# Question 3: Recovery -- % change from 2019 to latest year
nights_2019 = nuts3_clean[nuts3_clean["year"] == 2019].set_index("nuts_code")["nights"]
nights_now  = nuts3_clean[nuts3_clean["year"] == latest_year].set_index("nuts_code")["nights"]

# Only regions with data in both years
common = nights_2019.index.intersection(nights_now.index)
change = ((nights_now[common] - nights_2019[common]) / nights_2019[common] * 100).round(1)

recovery_df = change.reset_index()
recovery_df.columns = ["nuts_code", "pct_change"]

name_map = nuts3_clean[["nuts_code","region_name"]].drop_duplicates().set_index("nuts_code")["region_name"]
recovery_df["region_name"] = recovery_df["nuts_code"].map(name_map)
recovery_df["country"]     = recovery_df["nuts_code"].str[:2]
recovery_df = recovery_df.sort_values("pct_change", ascending=False)

print(f"Regions with data in both 2019 and {latest_year}: {len(recovery_df)}")
print()
print("=== Top 10 best-recovering regions ===")
print(recovery_df.head(10)[["country","region_name","pct_change"]].to_string(index=False))
print()
print(f"=== Top 10 regions still below 2019 levels ===")
print(recovery_df.tail(10)[["country","region_name","pct_change"]].to_string(index=False))

Regions with data in both 2019 and 2024: 1178

=== Top 10 best-recovering regions ===
country      region_name  pct_change
     BE     Arr. Waremme       411.2
     RO           Braila       310.3
     RO         Ialomita       254.2
     BG           Vratsa       206.6
     DE         Diepholz       202.6
     DE Nienburg (Weser)       188.1
     DK      Østsjælland       174.3
     PL        Chojnicki       170.3
     DE       Ravensburg       136.1
     DE       St. Wendel       135.1

=== Top 10 regions still below 2019 levels ===
country                         region_name  pct_change
     DE                            Stormarn       -62.4
     BE                    Arr. Dendermonde       -66.5
     DK                            Bornholm       -66.5
     BE                       Arr. Soignies       -66.7
     RO                           Teleorman       -72.7
     MT Gozo and Comino / Ghawdex u Kemmuna       -72.9
     IT                               Rieti       -74.8
     SI    

## 3. Export CSVs for Datawrapper

Four output files:

| File | Chart type | What it shows |
|------|------------|---------------|
| `eu_trend.csv` | Line chart | COVID crash and recovery arc, 2019-2024 |
| `map_nights_2024.csv` | Choropleth map NUTS3 | Nights by region, 2024 snapshot |
| `top20_regions.csv` | Bar chart | Top 20 regions by nights, 2024 |
| `recovery_change.csv` | Choropleth map NUTS3 | % change 2019 to 2024 by region |

**Datawrapper notes:**
- For NUTS3 choropleth maps: New Map -> Europe -> NUTS3 regions.
  The `nuts_code` column maps directly -- no ISO3 conversion needed.
- Nights values are kept in millions for readable chart labels.
- The bar chart should be sorted descending by `nights_millions`.

In [7]:
# CSV 1: EU trend
eu_out = eu_trend[["year","nights_bn"]].copy()
eu_out.columns = ["year", "nights_billion"]
eu_out.to_csv(OUTPUT_DIR / "eu_trend.csv", index=False)
print("✓ eu_trend.csv  ->  line chart")
print(eu_out.to_string(index=False))

✓ eu_trend.csv  ->  line chart
 year  nights_billion
 2019            2.91
 2020            1.45
 2021            1.90
 2022            2.86
 2023            3.05
 2024            3.12


In [8]:
# CSV 2: NUTS3 choropleth map, 2024
map_df = nuts3_clean[nuts3_clean["year"] == latest_year][["nuts_code","region_name","country","nights"]].copy()
map_df["nights_millions"] = (map_df["nights"] / 1e6).round(2)
map_df = map_df.drop(columns=["nights"]).sort_values("nuts_code")
map_df.to_csv(OUTPUT_DIR / "map_nights_2024.csv", index=False)
print(f"✓ map_nights_2024.csv  ->  choropleth map ({len(map_df)} regions)")
print(map_df.head(5).to_string(index=False))

✓ map_nights_2024.csv  ->  choropleth map (1204 regions)
nuts_code             region_name country  nights_millions
    AT111        Mittelburgenland      AT             0.32
    AT112          Nordburgenland      AT             1.71
    AT113           Südburgenland      AT             0.96
    AT121 Mostviertel-Eisenwurzen      AT             0.72
    AT122    Niederösterreich-Süd      AT             1.18


In [9]:
# CSV 3: Top 20 regions bar chart
top20_out = top20[["nuts_code","region_name","country","nights","share_pct"]].copy()
top20_out["nights_millions"] = (top20_out["nights"] / 1e6).round(1)
top20_out = top20_out.drop(columns=["nights"]).reset_index(drop=True)
top20_out.to_csv(OUTPUT_DIR / "top20_regions.csv", index=False)
print("✓ top20_regions.csv  ->  bar chart")
print(top20_out.to_string(index=False))

✓ top20_regions.csv  ->  bar chart
nuts_code                     region_name country  share_pct  nights_millions
    ES532                        Mallorca      ES        1.8             55.3
    ITI43                            Roma      IT        1.5             47.2
    FR101                           Paris      FR        1.4             43.2
    ES511                       Barcelona      ES        1.3             41.2
    ITH35                         Venezia      IT        1.2             38.8
    ITH10                   Bolzano-Bozen      IT        1.2             37.0
    ES709                        Tenerife      ES        1.1             35.8
    ES521              Alicante / Alacant      ES        1.0             32.6
    ES300                          Madrid      ES        1.0             32.3
    ES617                          Málaga      ES        1.0             30.5
    DE300                          Berlin      DE        1.0             30.4
    EL421 Kalymnos, Karpathos

In [10]:
# CSV 4: Recovery choropleth map
recovery_out = recovery_df[["nuts_code","region_name","country","pct_change"]].copy()
recovery_out.to_csv(OUTPUT_DIR / "recovery_change.csv", index=False)
print(f"✓ recovery_change.csv  ->  choropleth map ({len(recovery_out)} regions)")
print(recovery_out.head(5).to_string(index=False))

✓ recovery_change.csv  ->  choropleth map (1178 regions)
nuts_code  region_name country  pct_change
    BE334 Arr. Waremme      BE       411.2
    RO221       Braila      RO       310.3
    RO315     Ialomita      RO       254.2
    BG313       Vratsa      BG       206.6
    DE922     Diepholz      DE       202.6


In [11]:
# Summary
print("=== Output files ===")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    kb = f.stat().st_size / 1024
    print(f"  {f.name:<35} {kb:.1f} KB")

print()
print("Next steps:")
print("  eu_trend.csv          ->  Datawrapper: New Chart -> Lines")
print("  map_nights_2024.csv   ->  Datawrapper: New Map -> Europe NUTS3")
print("  top20_regions.csv     ->  Datawrapper: New Chart -> Bars")
print("  recovery_change.csv   ->  Datawrapper: New Map -> Europe NUTS3")

=== Output files ===
  eu_trend.csv                        0.1 KB
  map_nights_2024.csv                 33.4 KB
  recovery_change.csv                 33.1 KB
  top20_regions.csv                   0.7 KB

Next steps:
  eu_trend.csv          ->  Datawrapper: New Chart -> Lines
  map_nights_2024.csv   ->  Datawrapper: New Map -> Europe NUTS3
  top20_regions.csv     ->  Datawrapper: New Chart -> Bars
  recovery_change.csv   ->  Datawrapper: New Map -> Europe NUTS3
